In [16]:
from copy import copy
import os
from pathlib import Path
from typing import List, Tuple
from uuid import uuid4

import geopandas as gpd
import gtfs_kit as gk
import networkx as nx
import numpy as np
import pandas as pd
import partridge as ptg
from joblib import Parallel, delayed
from networkx.utils import pairwise
from rtree import index
from scipy.stats import hmean
from shapely.geometry import LineString, Point
from shapely.ops import substring


In [17]:
DATABASE = os.environ.get('DB_FOLDER')
DATABASE = Path(DATABASE) / 'beaga'

files =[
    f"{DATABASE}/GTFSBHTRANS.zip",
    ]

In [18]:
feeds = [
    gk.feed.read_feed(f, dist_units='m')
    for f in files
]

# Filter by Time Window

In [19]:
def filter_feed_by_time(feed,
                        start_time: str,
                        end_time: str):
    """
    Filters a Feed object to keep trips with stop_times in the window,
    cascading filters to related tables.
    Returns a new Feed object with updated attributes.
    """
    st = feed.stop_times
    trips = feed.trips

    mask = st["departure_time"].between(start_time, end_time)
    st_filt = st[mask]

    trip_ids = set(st_filt["trip_id"])
    trips_filt = trips[trips["trip_id"].isin(trip_ids)]

    route_ids = set(trips_filt["route_id"])
    svc_ids = set(trips_filt["service_id"])
    shape_ids = (set(trips_filt["shape_id"])
                 if "shape_id" in trips_filt else set())

    def filt(tbl, col, ids):
        df = getattr(feed, tbl, None)
        if df is None:
            return None
        if df.empty:
            return df
        return df[df[col].isin(ids)]

    # Clone the feed object
    filtered_feed = copy(feed)

    # Assign filtered tables
    filtered_feed.stop_times = st_filt
    filtered_feed.trips = trips_filt
    filtered_feed.routes = filt("routes", "route_id", route_ids)
    filtered_feed.calendar = filt("calendar", "service_id", svc_ids)
    filtered_feed.calendar_dates = filt("calendar_dates",
                                        "service_id", svc_ids)
    filtered_feed.shapes = filt("shapes", "shape_id", shape_ids)

    return filtered_feed


In [20]:
filtered_feed = filter_feed_by_time(
    feeds[0],
    start_time="06:00:00",
    end_time="08:00:00"
)

In [21]:
FOLDER = os.environ.get('OUT_FOLDER')

OUT_DIR = Path(FOLDER) / 'B'
OUT_DIR.mkdir(parents=True, exist_ok=True)

outpath = f"{OUT_DIR}/gtfs_morning_peak.zip"
filtered_feed.to_file(outpath)